# Liquidity Event Study: exploration

Visual companion to `docs/liquidity_event_study.md`. It reads the tables written by `scripts/run_study.py` and re-simulates seed 0 to plot a single run. All logic lives in `src/`; nothing here is a source of truth.

**Framing:** synthetic order flow with a designed stress scenario (DECISIONS.md ADR-002). This is not a real market event.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.event_study import StudyConfig, simulate

RESULTS = ROOT / "results"

## Pre-registered effects (stressed minus same-seed control, 20 seeds)

In [ ]:
effects = pd.read_csv(RESULTS / "event_study_effects.csv")
effects.round(2)

In [ ]:
pd.read_csv(RESULTS / "event_study_recovery.csv").round(1)

In [ ]:
pd.read_csv(RESULTS / "robustness_grid.csv").round(2)

## One run up close: seed 0, primary scenario

Re-simulated from its seed (about 2 s), so the plot is regenerated here rather than read from a stored output.

In [ ]:
cfg = StudyConfig()
_, stressed = simulate(cfg, seed=0, stressed=True)
_, control = simulate(cfg, seed=0, stressed=False)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ax, col in zip(axes, ["spread", "bid_depth_1", "mid"]):
    ax.axvspan(cfg.event_start / 60, cfg.event_end / 60, color="0.92")
    for df, color, label in ((control, "#8c8c86", "control"), (stressed, "#2a78d6", "stressed")):
        d = df[(df.t >= 3000) & (df.t <= 4800)]
        ax.plot(d.t / 60, d[col], color=color, linewidth=1, label=label)
    ax.set_ylabel(col)
axes[0].legend(frameon=False)
axes[-1].set_xlabel("simulated minutes")
fig.suptitle("Seed 0, synthetic flow: sell market orders x5 for 5 minutes (shaded)")
plt.show()

## Fill comparison by queue ahead

In [ ]:
pd.read_csv(RESULTS / "fill_comparison_queue_buckets.csv").round(3)